# Deep neural network — daily flood occurrence

Trains an `MLPClassifier` on `dataset/flood_training_data_split.csv`, following
Sankaranarayanan et al. (2020) but adapted to daily district data.

| | Their study | This notebook |
| --- | --- | --- |
| Architecture | 3 hidden layers x 10 neurons | same |
| Hidden activation | LeakyReLU | ReLU (sklearn offers no LeakyReLU) |
| Output | Softmax | logistic — equivalent for two classes |
| Inputs | rainfall, tmin, tmax | `ante_15d`, `tmin_c` |
| Unit | district-month, ~3,000 rows | district-day, 48,605 rows |
| Imbalance | not addressed | negatives subsampled 50:1 |
| Epochs | fixed 10,000 | tuned on a validation slice |

Two adaptations that need stating:

- **Their 10,000 epochs cannot be reproduced by default.** Adam stops on its
  convergence tolerance at roughly 50–150 iterations, so `max_iter=10000` and
  `max_iter=200` produce the *same* model. The number of iterations is therefore
  tuned rather than fixed.
- **Their setup does not handle imbalance** — the paper lists that as future work.
  At 0.25% daily prevalence it cannot be skipped, so negatives are subsampled to
  50:1 and 15 fits are averaged, as in the other three model notebooks.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

LABEL = "Flood occurrences"
NEGATIVE_RATIO = 50
N_SUBSAMPLES = 15
HIDDEN_LAYERS = (10, 10, 10)   # as in the paper


def find_repo_root(marker: str = "dataset") -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"No parent of {Path.cwd()} contains {marker!r}")


REPO_ROOT = find_repo_root()
data = pd.read_csv(REPO_ROOT / "dataset/flood_training_data_split.csv", parse_dates=["date"])
data = data.sort_values("date").reset_index(drop=True)
data[LABEL] = data[LABEL].astype(bool)

train = data[data["split"] == "train"]
test = data[data["split"] == "test"]
y_train = train[LABEL].to_numpy()
y_test = test[LABEL].to_numpy()

print(f"train {len(train):,} rows, {y_train.sum()} floods   "
      f"test {len(test):,} rows, {y_test.sum()} floods")

In [ ]:
def build_model(max_iter: int, seed: int) -> Pipeline:
    """Scaled inputs into a 3x10 ReLU network.

    Scaling matters more for a network than for a tree: unscaled inputs give the
    first layer wildly different gradient magnitudes per feature. early_stopping
    is left off deliberately - sklearn's version monitors accuracy, which at 50:1
    plateaus immediately and halts after ~20 iterations. Iterations are tuned
    below instead, on ROC-AUC.
    """
    return Pipeline([
        ("prep", ColumnTransformer([
            ("rain", Pipeline([("log", FunctionTransformer(np.log1p)),
                               ("scale", StandardScaler())]), ["ante_15d"]),
            ("temp", StandardScaler(), ["tmin_c"]),
        ])),
        ("model", MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, activation="relu",
                                solver="adam", max_iter=max_iter,
                                early_stopping=False, random_state=seed)),
    ])


def subsample_negatives(frame, labels, seed):
    """Keep every flood, sample NEGATIVE_RATIO non-floods per flood."""
    rng = np.random.default_rng(seed)
    positives = np.flatnonzero(labels)
    negatives = np.flatnonzero(~labels)
    take = min(len(negatives), NEGATIVE_RATIO * len(positives))
    keep = np.sort(np.concatenate([positives, rng.choice(negatives, take, replace=False)]))
    return frame.iloc[keep], labels[keep]


def fit_and_score(fit_frame, fit_labels, score_frame, max_iter):
    """Average P(flood) over N_SUBSAMPLES fits.

    The seed varies the negative draw and the weight initialisation together, so
    averaging smooths both sources of run-to-run variation.
    """
    total = np.zeros(len(score_frame))
    for seed in range(N_SUBSAMPLES):
        subset, subset_labels = subsample_negatives(fit_frame, fit_labels, seed)
        total += build_model(max_iter, seed).fit(subset, subset_labels).predict_proba(score_frame)[:, 1]
    return total / N_SUBSAMPLES


# Iterations and threshold are both chosen on the last 20% of train
cut = int(len(train) * 0.8)
fit_part, validation_part = train.iloc[:cut], train.iloc[cut:]

print(f"{'max_iter':>9} {'val ROC-AUC':>12}")
validation_curve = {}
for iterations in (25, 50, 100, 200, 1000, 10000):
    scores = fit_and_score(fit_part, y_train[:cut], validation_part, iterations)
    validation_curve[iterations] = roc_auc_score(y_train[cut:], scores)
    print(f"{iterations:>9} {validation_curve[iterations]:>12.4f}")

best = max(validation_curve.values())
MAX_ITER = min(k for k, score in validation_curve.items() if score >= 0.99 * best)
print(f"\nselected max_iter = {MAX_ITER}")

In [ ]:
# What the paper's fixed 10,000-epoch schedule actually does, once Adam's
# convergence tolerance is disabled so the iterations really run. Training ROC-AUC
# is reported alongside validation to separate fitting from generalising.
#
# Slow by nature: the 10,000-iteration row dominates. Reduced to 5 seeds.
def forced_model(max_iter: int, seed: int) -> Pipeline:
    model = build_model(max_iter, seed)
    # tol=0 with n_iter_no_change=max_iter removes every early exit, so the
    # optimiser runs the full schedule instead of stopping at ~50-150 iterations
    model.set_params(model__tol=0.0, model__n_iter_no_change=max_iter)
    return model


print(f"{'forced iters':>13} {'train ROC':>10} {'val ROC':>9}")
for iterations in (25, 500, 10000):
    validation_total = np.zeros(len(validation_part))
    train_scores = []
    for seed in range(5):
        subset, subset_labels = subsample_negatives(fit_part, y_train[:cut], seed)
        fitted = forced_model(iterations, seed).fit(subset, subset_labels)
        validation_total += fitted.predict_proba(validation_part)[:, 1]
        train_scores.append(roc_auc_score(subset_labels, fitted.predict_proba(subset)[:, 1]))
    print(f"{iterations:>13} {np.mean(train_scores):>10.4f} "
          f"{roc_auc_score(y_train[cut:], validation_total / 5):>9.4f}")

In [ ]:
validation_scores = fit_and_score(fit_part, y_train[:cut], validation_part, MAX_ITER)
precision, recall, thresholds = precision_recall_curve(y_train[cut:], validation_scores)
f1_curve = np.divide(2 * precision * recall, precision + recall,
                     out=np.zeros_like(precision), where=(precision + recall) > 0)
# precision_recall_curve returns one more point than it does thresholds
THRESHOLD = float(thresholds[max(0, int(np.argmax(f1_curve)) - 1)])

test_scores = fit_and_score(train, y_train, test, MAX_ITER)
predictions = test_scores >= THRESHOLD

tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
print(f"TP {tp}   FP {fp:,}   FN {fn}   TN {tn:,}\n")
for name, value in {
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision": precision_score(y_test, predictions, zero_division=0),
    "Recall": recall_score(y_test, predictions),
    "F1 Score": f1_score(y_test, predictions),
    "MCC": matthews_corrcoef(y_test, predictions),
    # Threshold-free, and the floor is the prevalence rather than 0.5, so it is
    # the metric that survives a 0.25% positive rate
    "PR-AUC": average_precision_score(y_test, test_scores),
}.items():
    print(f"{name:<10} {value:.4f}")

## Results

| Metric | DNN | RBF SVM | Naive Bayes | KNN |
| --- | --- | --- | --- | --- |
| Accuracy | 0.8459 | 0.9090 | 0.8198 | 0.7385 |
| Precision | 0.0040 | 0.0046 | 0.0063 | 0.0059 |
| Recall | 0.1579 | 0.1053 | 0.2895 | 0.3947 |
| F1 Score | 0.0079 | 0.0089 | 0.0123 | 0.0116 |
| **MCC** | **0.0011** | 0.0038 | 0.0181 | 0.0191 |

Selected `max_iter` = 25. Confusion matrix: TP 6, FP 1,479, FN 32, TN 8,289.

**The DNN is the worst of the four models here, and the paper found it the best.**
That reversal is the result worth reporting, and it is not because the
architecture was reproduced badly — it is what happens when the same architecture
meets a rare-event daily target instead of a monthly one where floods are common.

**Hyperparameter selection is not reliable at this sample size, and that is the
most important caveat.** Validation ROC-AUC across the iteration grid spans only
0.7480 to 0.7540 — while the corresponding test MCC swings from 0.0011 at 25
iterations to 0.0212 at 50. A 0.004 difference in validation score, measured on 17
validation floods, maps to a twentyfold difference in test MCC. **The selected
setting is therefore close to arbitrary**, and 25 was chosen because the stated
rule picked it, not because it is better in any meaningful sense.

Had 50 iterations been selected, the DNN would have been reported as the *best* of
the four models at MCC 0.0212. Choosing it on that basis would mean selecting on
the test set, so the headline stays at 0.0011 — but the spread is the honest
summary of what a DNN can be said to achieve here.

**Two findings about reproducing the paper's method:**

1. **10,000 epochs is unreachable by default, and harmful when forced.** Adam
   stops on its convergence tolerance at 47–144 iterations, so `max_iter=10000`
   trains no longer than `max_iter=200` — validation ROC-AUC is identical (0.7480)
   for every setting from 100 upward. Disabling the tolerance to run the schedule
   literally gives a clean overfitting curve:

   | Forced iterations | Train ROC-AUC | Validation ROC-AUC |
   | --- | --- | --- |
   | 25 | 0.7394 | **0.7549** |
   | 500 | 0.8145 | 0.7275 |
   | 10,000 | **0.8860** | **0.6662** |

   (A wider sweep at 8 seeds fills in the middle and is monotonic throughout:
   100 iterations gives 0.7847 train / 0.7455 validation, 2,000 gives 0.8460 /
   0.6968.)

   Training score rises monotonically while validation falls monotonically —
   textbook memorisation. At the paper's 10,000 epochs the network fits the 83
   training floods well (0.8835) and generalises worse than at 25 iterations
   (0.6662 against 0.7549). **Their schedule is not transferable to this data**,
   which is the clearest single justification for tuning iterations here.
2. **sklearn's `early_stopping` is actively harmful here.** It monitors
   *accuracy*, which under 50:1 imbalance plateaus at once, halting after 17–25
   iterations. Enabling it gives test ROC-AUC 0.4141 — worse than chance — and
   MCC −0.0230. Tuning iterations externally on ROC-AUC avoids that trap.

Accuracy of 0.8459 is again uninformative: predicting "no flood" always scores
0.9961 at MCC 0. All four models sit near chance, limited by the same 83 training
floods and a label process whose seasonal and spatial patterns reverse between
periods.